# **LangChain Loader, Splitter, Embeddings, and VectorStore**

### **Objective:**  
To implement LangChain’s loaders, text splitters, embeddings, and vector stores for efficient document loading, processing, embedding, and similarity search. This activity demonstrates how to handle text and PDF data, create embeddings using Hugging Face and OpenAI models, and perform similarity search using FAISS.

---

### **Note:**  
- Before running any demo, ensure that the **requirements.txt** file is installed. This file contains all the required dependencies for **all demos and guided practices under Building LLM Applications**.
- If the dependencies were already installed earlier (after creating the virtual environment), there is no need to install them again. You can directly proceed with running the demo.
- Refer to Lesson_01 **Demo_01_Zero_Shot_Prompting.ipynb** Step 1 for creating a virtual environment and installing the requirements.txt 
- Ensure you select the right kernel **Python (myenv)** while running the demos  
- You will need the following files in your working directory:  
  - `state_of_union.txt` (a sample text document)  
  - `michael_resume.pdf` (a sample PDF resume file)  

---

### **Steps to perform:**
1. Import the necessary modules  
2. Load text data from a file using TextLoader  
3. Load PDFs from the internet using PyPDFLoader  
4. Split the documents using RecursiveCharacterTextSplitter  
5. Embed the documents using HuggingFaceEmbeddings and print the embedding length  
6. Embed the documents using OpenAIEmbeddings and print the embedding length  
7. Create a FAISS instance  
8. Perform a similarity search on the FAISS instance  
9. Persist the FAISS instance  
10. Load the persisted FAISS instance  

---


In [1]:
!pip install torch==2.1.2

     |████████████████████████████████| 670.2 MB 6.1 kB/s  eta 0:00:013��█▍            | 406.4 MB 91.6 MB/s eta 0:00:03
  Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
     |████████████████████████████████| 14.1 MB 86.8 MB/s eta 0:00:01
     |████████████████████████████████| 731.7 MB 25 kB/s s eta 0:00:012    |████████████████▎               | 373.4 MB 71.3 MB/s eta 0:00:06
     |████████████████████████████████| 823 kB 71.5 MB/s eta 0:00:01
     |████████████████████████████████| 203 kB 74.1 MB/s eta 0:00:01
     |████████████████████████████████| 23.7 MB 55.8 MB/s eta 0:00:01�████▍                   | 9.1 MB 55.8 MB/s eta 0:00:01
     |████████████████████████████████| 410.6 MB 9.9 kB/s  eta 0:00:012   |████▉                           | 61.8 MB 77.0 MB/s eta 0:00:05
     |████████████████████████████████| 56.5 MB 60.7 MB/s eta 0:00:01
  Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
     |████████████████████████████████| 209.8 MB 5.1 kB/s s eta 0:00:01
    

### **Step 1: Import the Necessary Modules**
- Import essential modules from LangChain for document loading, text splitting, embeddings, and vector storage






In [2]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import faiss

import warnings
warnings.filterwarnings("ignore")

### **Step 2: Load text data from a file using TextLoader**

- Load a plain text file (state_of_union.txt) and print the first 100 characters to confirm successful loading




In [3]:
text_loader = TextLoader("state_of_union.txt")
text_document = text_loader.load()
print(text_document[:100])  # Prints the first 100 characters of the text document

[Document(metadata={'source': 'state_of_union.txt'}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determinatio

### **Step 3: Load PDFs from the internet using PyPDFLoader**
- Load a PDF document (e.g., michael_resume.pdf) and split it into pages.
- Print the first page’s content to verify loading.


In [4]:
pdf_loader = PyPDFLoader("michael_resume.pdf")
pdf_pages = pdf_loader.load_and_split()
print(pdf_pages[0])  # Prints the first 100 characters of the first page of the PDF


page_content='CURRICULUM VITAE :  
M ichael M . Scott OBE, B.Sc., Dip.Ed  
 
Home address:  Strome House     Date of Birth: 10.5.51 
   North Strome     Place of Birth: Edinburgh 
   Lochcarron     Married to Sue Scott; 2 stepchildren 
   Ross-shire, IV54 8YJ 
Telephone (work): 01520 722901     Website: www.mmscott.co.uk 
Telephone (home):  01520 722588    E-mail:  MSStrome@aol.com 
 
Awarded OBE in Queen’s Birthday Honours, June 2005, “for services to biodiversity conservation in 
Scotland”. 
Awarded Planta Europa ‘Silver Lead’ Award in September 2007, “for excellent work in European wild plant 
conservation”. 
 
Education 
Primary education: George Heriots School, Edinburgh (1956-1962) 
Secondary education:  Madras College, St Andrews (1962-69). 
Further education: University of Aberdeen (1969-1974): 
    Bachelor of Science (Honours; upper second) in Botany, 1973 
    Diploma of Education, 1974 
   Aberdeen College of Education (1973 - 1974): 
    Certificate of Education in seconda

### **Step 4: Split the documents using RecursiveCharacterTextSplitter**
- Split the loaded PDF into smaller text chunks for processing


In [5]:
doc_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=64)
split_texts = doc_splitter.split_documents(pdf_pages)
print(len(split_texts))  # Prints the number of chunks the PDF has been split into


15


In [6]:
# Print number of chunks created
print(f"Splitted into {len(split_texts)} chunks.")
print()

# Print first chunk for inspection
print(split_texts[0])

Splitted into 15 chunks.

page_content='CURRICULUM VITAE :  
M ichael M . Scott OBE, B.Sc., Dip.Ed  
 
Home address:  Strome House     Date of Birth: 10.5.51 
   North Strome     Place of Birth: Edinburgh 
   Lochcarron     Married to Sue Scott; 2 stepchildren 
   Ross-shire, IV54 8YJ 
Telephone (work): 01520 722901     Website: www.mmscott.co.uk 
Telephone (home):  01520 722588    E-mail:  MSStrome@aol.com 
 
Awarded OBE in Queen’s Birthday Honours, June 2005, “for services to biodiversity conservation in 
Scotland”. 
Awarded Planta Europa ‘Silver Lead’ Award in September 2007, “for excellent work in European wild plant 
conservation”. 
 
Education 
Primary education: George Heriots School, Edinburgh (1956-1962) 
Secondary education:  Madras College, St Andrews (1962-69). 
Further education: University of Aberdeen (1969-1974): 
    Bachelor of Science (Honours; upper second) in Botany, 1973 
    Diploma of Education, 1974 
   Aberdeen College of Education (1973 - 1974):' metadata={'so

- Splitting large documents into smaller chunks helps in efficient embedding and retrieval operations.

### **Step 5: Embed the documents using HuggingFaceEmbeddings and print the length of the embedding**
- Use the Hugging Face embedding model to convert text chunks into numerical vectors



In [7]:
# MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
# hf_embed = HuggingFaceEmbeddings(model_name=MODEL_NAME)
# text = split_texts[0].page_content
# hf_embed_result = hf_embed.embed_documents([text])
# print(len(hf_embed_result[0]))  # Prints the length of the first embedded document

In [8]:
texts = [doc.page_content for doc in split_texts]

### **Step 6: Embed the documents using OpenAIEmbeddings and print the length of the embedding**
- Use OpenAIEmbeddings for generating embeddings based on OpenAI’s API


In [9]:
openai_embed = OpenAIEmbeddings()
openai_embed_result = openai_embed.embed_documents(texts)
print(len(openai_embed_result[0]))  # Prints the length of the first embedded document


1536


In [10]:
print(openai_embed.model)

text-embedding-ada-002


In [11]:
# Print the embedding of the 0th chunk
print(openai_embed.embed_query(split_texts[0].page_content))

# # Print the length of the 0th embedding
# print(len(openai_embed.embed_query(split_texts[0].page_content)))

# # Print the length of the 110th embedding
# print(len(openai_embed.embed_query(split_texts[10].page_content)))

[-0.025390451413754882, -0.014135300912573387, -0.023704833946067958, -0.01094324498743272, -0.01248286460510408, 0.06280586779413974, 0.009177991058842903, 0.011898871342396693, -0.015290016091488185, -0.008580725518312982, 0.0008676951278756162, -0.003656596824800117, 0.009695622001115055, 0.020028328238872756, -0.007319829788511502, 0.01039243288494295, 0.021037045567771993, 0.006895107119303496, 0.005020354803202005, -0.006941561023004929, -0.027686608562439385, 0.011945324780436844, 0.001471597680431706, -0.01044552292755565, 0.01368403322205903, -0.005727119662566162, 0.015329833856278351, -0.011786053721276182, 0.0026561757172778493, -0.037216323831143795, -0.001503120038752147, -0.002088772801848631, 0.005299079156733164, -0.023638471625632724, -0.03859667425229961, -0.014427298475249642, -0.007658280789227909, -0.005607666368195666, 0.050541996238768774, -0.0036167790600099515, 0.02032032580154901, 0.016962362212675136, -0.013723850986849197, -0.010525158457135983, 0.007266739

- Embeddings convert textual content into fixed-size dense vectors, useful for similarity search and clustering.

### **Step 7: Create a FAISS instance**
- Create a FAISS (Facebook AI Similarity Search) vector store using the embedded text data

In [12]:
# Create FAISS instance from documents and embeddings
faiss_index = FAISS.from_documents(split_texts, openai_embed)


- FAISS efficiently indexes high-dimensional vectors for similarity retrieval.

### **Step 8: Perform a similarity search on the FAISS instance**
- Perform a similarity search to find documents related to a specific query (e.g., “candidate’s skill sets”)

In [13]:
# Perform a similarity search and print the top two most similar documents
search_result = faiss_index.similarity_search_with_score("What is the candidate's skill sets?", k=2)
for i, (doc, score) in enumerate(search_result, start=1):
    print(f"\n=== Result {i} | Score: {score:.4f} ===")
    print(doc.page_content.strip())



=== Result 1 | Score: 0.4209 ===
Computer knowledge 
I am reasonably fluent in basic PC computer skills, using Windows XP, Word, WordPro, Excel, PowerPoint, 
Adobe Photoshop Elements, e-mail, internet etc.  I have full computer and broadband facilities at home. 
 
Other interests 
Botanising (especially mountain flowers), travel, walking, Scottish islands, gardening, photography, 
computers, rugby supporter, cinema, good wine, Runrig concerts (!). 
 
[updated, 26.03.08]

=== Result 2 | Score: 0.4809 ===
CURRICULUM VITAE :  
M ichael M . Scott OBE, B.Sc., Dip.Ed  
 
Home address:  Strome House     Date of Birth: 10.5.51 
   North Strome     Place of Birth: Edinburgh 
   Lochcarron     Married to Sue Scott; 2 stepchildren 
   Ross-shire, IV54 8YJ 
Telephone (work): 01520 722901     Website: www.mmscott.co.uk 
Telephone (home):  01520 722588    E-mail:  MSStrome@aol.com 
 
Awarded OBE in Queen’s Birthday Honours, June 2005, “for services to biodiversity conservation in 
Scotland”. 
Award

- The FAISS instance returns the most relevant chunks based on cosine similarity scores.

### **Step 9: Persist the FAISS instance**
- Save the FAISS index locally for reuse or sharing

In [14]:
# Save the FAISS index to a file
faiss_index.save_local("faiss_index")


### __Step 10: Load the persisted FAISS instance__
- Reload the FAISS index safely and verify that the stored embeddings can still perform similarity search

In [16]:
# Load the persisted FAISS index from the file safely
faiss_index_loaded = FAISS.load_local(
    "faiss_index", 
    openai_embed,
    allow_dangerous_deserialization=True
)

# Perform a similarity search with the loaded FAISS index
vector_search_result = faiss_index_loaded.similarity_search_with_score(
    "What is the candidate's skill sets?", 
    k=2
)
for i, (doc, score) in enumerate(vector_search_result, start=1):
    print(f"\n=== Result {i} | Score: {score:.4f} ===")
    print(doc.page_content.strip())


=== Result 1 | Score: 0.4209 ===
Computer knowledge 
I am reasonably fluent in basic PC computer skills, using Windows XP, Word, WordPro, Excel, PowerPoint, 
Adobe Photoshop Elements, e-mail, internet etc.  I have full computer and broadband facilities at home. 
 
Other interests 
Botanising (especially mountain flowers), travel, walking, Scottish islands, gardening, photography, 
computers, rugby supporter, cinema, good wine, Runrig concerts (!). 
 
[updated, 26.03.08]

=== Result 2 | Score: 0.4809 ===
CURRICULUM VITAE :  
M ichael M . Scott OBE, B.Sc., Dip.Ed  
 
Home address:  Strome House     Date of Birth: 10.5.51 
   North Strome     Place of Birth: Edinburgh 
   Lochcarron     Married to Sue Scott; 2 stepchildren 
   Ross-shire, IV54 8YJ 
Telephone (work): 01520 722901     Website: www.mmscott.co.uk 
Telephone (home):  01520 722588    E-mail:  MSStrome@aol.com 
 
Awarded OBE in Queen’s Birthday Honours, June 2005, “for services to biodiversity conservation in 
Scotland”. 
Award

In [ ]:
print(torch.__version__)
print(transformers.__version__)
print(sentence_transformers.__version__)

- This ensures persistence of vector indexes for scalable retrieval in production environments.

### **Conclusion**

By following these steps, you have successfully implemented LangChain’s Loader, Splitter, Embeddings, and VectorStore functionalities. You learned how to load text and PDF data, split documents into manageable chunks, embed text using different models, store vectors in FAISS, and perform similarity searches for efficient document retrieval.

---